In [1]:
import requests

PEXELS_API_KEY = "OzA5NAaBNk7otiMnUDvQS4ZwodwYraXZHHzw5FDV05WxGZTO6T1UmJN3"
url = "https://api.pexels.com/v1/search?query=construction"

headers = {"Authorization": PEXELS_API_KEY}
response = requests.get(url, headers=headers)

# Extract rate limit headers
rate_limit = response.headers.get("X-Ratelimit-Limit")
remaining = response.headers.get("X-Ratelimit-Remaining")
reset_time = response.headers.get("X-Ratelimit-Reset")

print(f"Rate Limit: {rate_limit}, Remaining: {remaining}, Reset Time: {reset_time}")


Rate Limit: 25000, Remaining: 23656, Reset Time: 1752376841


In [2]:
import os

# 📂 Set transfer learning folder
base_path = r"E:\yolo_project\transfer_learning"
os.makedirs(base_path, exist_ok=True)

# 🔄 Create dataset folders (images + labels)
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(base_path, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(base_path, split, "labels"), exist_ok=True)

print("✅ YOLO dataset folders created successfully!")


✅ YOLO dataset folders created successfully!


In [3]:
import torch.nn as nn
import torch

class FeatureFusion(nn.Module):
    def __init__(self):
        super(FeatureFusion, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        return x

fusion_model = FeatureFusion()
print(fusion_model)


FeatureFusion(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
)


This would extract additional spatial features, improving detection accuracy when merged with YOLO.

In [4]:
class SelfAttention(nn.Module):
    def __init__(self, in_dim):
        super(SelfAttention, self).__init__()
        self.query = nn.Conv2d(in_dim, in_dim // 8, 1)
        self.key = nn.Conv2d(in_dim, in_dim // 8, 1)
        self.value = nn.Conv2d(in_dim, in_dim, 1)

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        attention = torch.softmax(Q @ K.transpose(-2, -1), dim=-1)
        x = attention @ V
        return x

attention_module = SelfAttention(in_dim=128)
print(attention_module)


SelfAttention(
  (query): Conv2d(128, 16, kernel_size=(1, 1), stride=(1, 1))
  (key): Conv2d(128, 16, kernel_size=(1, 1), stride=(1, 1))
  (value): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1))
)


This prioritizes small objects like helmets, cables, and scaffolding while filtering irrelevant clutter.

In [5]:
import torch
from ultralytics import YOLO

# Ensure GPU usage
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.set_device(0)

# Load YOLO11 pretrained model
base_model = YOLO("yolo11n.pt").to(device)
print("✅ YOLO11 model loaded successfully.")


✅ YOLO11 model loaded successfully.


In [6]:
class EnhancedYOLO11(nn.Module):
    def __init__(self, base_model):
        super(EnhancedYOLO11, self).__init__()
        self.base_model = base_model
        self.fusion = FeatureFusion()
        self.attention = SelfAttention(in_dim=128)
        
    def forward(self, x):
        yolo_features = self.base_model(x)
        fused_features = self.fusion(yolo_features)
        enhanced_features = self.attention(fused_features)
        return enhanced_features

enhanced_model = EnhancedYOLO11(base_model).to(device)
print("✅ Enhanced YOLO11 model ready.")


✅ Enhanced YOLO11 model ready.


In [10]:
import os
import json

# 📂 Paths
dataset_path = r"E:\yolo_project\mined"  # Existing dataset
labels_path = r"E:\yolo_project\transfer_learning\labels"  # New annotation folder
os.makedirs(labels_path, exist_ok=True)

# 🔄 Convert bounding box annotations to YOLO format
def convert_annotation(image_name, category, split):
    json_file = image_name.replace(".jpg", ".json").replace(".png", ".json")
    json_path = os.path.join(dataset_path, split, category, json_file)

    if not os.path.exists(json_path):
        print(f"⚠️ No annotation file found for {image_name}. Skipping...")
        return None

    with open(json_path, "r") as f:
        data = json.load(f)

    img_width = data.get("imageWidth", 640)
    img_height = data.get("imageHeight", 640)

    yolo_annotations = []
    for obj in data["shapes"]:
        class_id = obj["label"]  # Ensure labels match class IDs
        x_min, y_min, x_max, y_max = obj["points"][0][0], obj["points"][0][1], obj["points"][1][0], obj["points"][1][1]

        # Convert bounding box to YOLO format
        x_center = ((x_min + x_max) / 2) / img_width
        y_center = ((y_min + y_max) / 2) / img_height
        width = (x_max - x_min) / img_width
        height = (y_max - y_min) / img_height

        yolo_annotations.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    return yolo_annotations

# 🔄 Process images
for split in ["train", "val", "test"]:
    split_path = os.path.join(dataset_path, split)
    if not os.path.exists(split_path):
        continue

    for category in os.listdir(split_path):
        category_path = os.path.join(split_path, category)
        if not os.path.isdir(category_path):
            continue

        for image_file in os.listdir(category_path):
            if image_file.endswith((".jpg", ".png")):
                annotations = convert_annotation(image_file, category, split)
                
                if annotations:
                    label_filename = image_file.replace(".jpg", ".txt").replace(".png", ".txt")
                    with open(os.path.join(labels_path, label_filename), "w") as f:
                        f.write("\n".join(annotations))

print("✅ YOLO annotations saved in E:\\yolo_project\\transfer_learning\\labels!")


⚠️ No annotation file found for construction crane_1.jpg. Skipping...
⚠️ No annotation file found for construction crane_2.jpg. Skipping...
⚠️ No annotation file found for construction crane_3.jpg. Skipping...
⚠️ No annotation file found for construction crane_4.jpg. Skipping...
⚠️ No annotation file found for construction_crane_train_1.jpg. Skipping...
⚠️ No annotation file found for construction_crane_train_2.jpg. Skipping...
⚠️ No annotation file found for construction_crane_train_3.jpg. Skipping...
⚠️ No annotation file found for construction_crane_train_4.jpg. Skipping...
⚠️ No annotation file found for construction_crane_train_5.jpg. Skipping...
⚠️ No annotation file found for construction_crane_train_6.jpg. Skipping...
⚠️ No annotation file found for construction_crane_train_7.jpg. Skipping...
⚠️ No annotation file found for construction_crane_train_8.jpg. Skipping...
⚠️ No annotation file found for construction_crane_train_9.jpg. Skipping...
⚠️ No annotation file found for cons

In [13]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import os
from PIL import Image

# 📂 Base path for mined dataset
base_path = r"E:\yolo_project\mined"
transfer_path = r"E:\yolo_project\transfer_learning\labels"  # ✅ Save labels here
os.makedirs(transfer_path, exist_ok=True)

# ✅ Define dataset loader
class YOLODataset(Dataset):
    def __init__(self, dataset_split, transform=None, save_labels=False):
        self.split_path = os.path.join(base_path, dataset_split)
        self.transform = transform
        self.save_labels = save_labels
        self.label_save_path = os.path.join(transfer_path, dataset_split)  # ✅ Save labels in transfer_learning
        os.makedirs(self.label_save_path, exist_ok=True)

        # 🔄 Collect all images + dynamically assign labels
        self.img_files = []
        self.labels = []
        for category in os.listdir(self.split_path):
            category_path = os.path.join(self.split_path, category)
            if not os.path.isdir(category_path):
                continue

            for img in os.listdir(category_path):
                if img.endswith((".jpg", ".png")):
                    self.img_files.append(os.path.join(category_path, img))
                    self.labels.append(category)  # ✅ Use folder name as label

        print(f"📊 Found {len(self.img_files)} images in {dataset_split}")

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        category_label = self.labels[idx]  # ✅ Folder name as label

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        # ✅ Save label as YOLO format
        label_file = os.path.join(self.label_save_path, os.path.basename(img_path).replace(".jpg", ".txt").replace(".png", ".txt"))
        if self.save_labels:
            with open(label_file, "w") as f:
                f.write(f"{category_label} 0.5 0.5 1.0 1.0")  # Placeholder bbox (fix later)

        return img, category_label  # Label can later be mapped to class IDs

# 🔄 Apply transformations
transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),
])

# ✅ Create dataset loaders
train_dataset = YOLODataset("train", transform, save_labels=True)
val_dataset = YOLODataset("val", transform, save_labels=True)
test_dataset = YOLODataset("test", transform, save_labels=True)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("✅ Dataset now correctly maps images to labels based on folder structure.")


📊 Found 658 images in train
📊 Found 432 images in val
📊 Found 191 images in test
✅ Dataset now correctly maps images to labels based on folder structure.


In [ ]:
import torch

# ✅ IoU Calculation in PyTorch
def siou_loss(pred_box, gt_box):
    """
    Compute SIoU loss between predicted and ground truth boxes.
    pred_box, gt_box: Tensor (x_center, y_center, width, height)
    """
    x_p, y_p, w_p, h_p = pred_box[..., 0], pred_box[..., 1], pred_box[..., 2], pred_box[..., 3]
    x_g, y_g, w_g, h_g = gt_box[..., 0], gt_box[..., 1], gt_box[..., 2], gt_box[..., 3]

    # Compute aspect ratio penalty
    aspect_ratio_pred = w_p / h_p
    aspect_ratio_gt = w_g / h_g
    aspect_ratio_loss = torch.abs(torch.log(aspect_ratio_pred / aspect_ratio_gt))

    # Compute IoU
    inter_x1 = torch.max(x_p - w_p / 2, x_g - w_g / 2)
    inter_y1 = torch.max(y_p - h_p / 2, y_g - h_g / 2)
    inter_x2 = torch.min(x_p + w_p / 2, x_g + w_g / 2)
    inter_y2 = torch.min(y_p + h_p / 2, y_g + h_g / 2)

    inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
    union_area = w_p * h_p + w_g * h_g - inter_area
    iou = inter_area / (union_area + 1e-6)

    # Compute SIoU loss (IoU + aspect ratio penalty)
    siou = iou - aspect_ratio_loss

    return 1 - siou  # Loss decreases as SIoU improves

print("✅ IoU function ready for bounding box evaluation.")


✅ IoU function ready for bounding box evaluation.


In [ ]:
import torch.optim as optim

# ✅ Ensure model is in training mode
enhanced_model.train()
enhanced_model.to(device)

# ✅ Define optimizer & scheduler
optimizer = optim.Adam(enhanced_model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)  # Learning rate decay

# 🔄 Training loop
for epoch in range(50):  # Adjust epochs as needed
    total_loss = 0

    for batch_idx, (images, labels) in enumerate(train_loader):  # ✅ Use enumerate for batch tracking
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = enhanced_model(images)
        
        # ✅ Ensure YOLO loss is correctly defined
        loss = yolo_loss(outputs, labels)  # Modify if using YOLO11's loss structure
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:  # ✅ Print progress every 10 batches
            print(f"Epoch {epoch+1}, Batch {batch_idx}: Loss = {loss.item():.4f}")

    scheduler.step()  # ✅ Adjust learning rate

    print(f"Epoch {epoch+1}: Avg Loss = {total_loss / len(train_loader):.4f}")


I will train bounding boxes later and first just worry about classification -> I dont have annotations which I have to do manually later.

In [33]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import os
from PIL import Image
from ultralytics import YOLO

# 📂 Base path for dataset
base_path = r"E:\yolo_project\mined"

# ✅ Ensure class mapping is correctly built from categories (not train/val/test)
class_mapping = {}
for split in ["train", "val", "test"]:
    split_path = os.path.join(base_path, split)
    if not os.path.exists(split_path):
        continue
    for category in os.listdir(split_path):
        category_path = os.path.join(split_path, category)
        if os.path.isdir(category_path):  # ✅ Ensure it's a folder, not a file
            if category not in class_mapping:
                class_mapping[category] = len(class_mapping)

print("✅ Verified Class Mapping:", class_mapping)

# ✅ Define dataset loader
class YOLODataset(Dataset):
    def __init__(self, dataset_split, transform=None):
        self.split_path = os.path.join(base_path, dataset_split)
        self.transform = transform
        self.img_files = []
        self.labels = []

        for category in os.listdir(self.split_path):
            category_path = os.path.join(self.split_path, category)
            if not os.path.isdir(category_path):
                continue
            for img in os.listdir(category_path):
                if img.endswith((".jpg", ".png")):
                    self.img_files.append(os.path.join(category_path, img))
                    self.labels.append(category)  # ✅ Use folder name as label

        print(f"📊 Found {len(self.img_files)} images in {dataset_split}")

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        category_label = self.labels[idx]

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img, category_label

# 🔄 Apply transformations
transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),
])

# ✅ Create dataset loaders
train_dataset = YOLODataset("train", transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)

# ✅ Load YOLO11 backbone for classification
class YOLOClassifier(torch.nn.Module):
    def __init__(self, base_model, num_classes):
        super(YOLOClassifier, self).__init__()
        self.base_model = base_model.model.model[:-1]  # Remove detection head
        self.classifier = torch.nn.Linear(1024, num_classes)  # Classification head

    def forward(self, x):
        print(f"🔍 Input Image Shape: {x.shape}")  # Debug step
        x = self.base_model(x)  # Feature extraction
        print(f"🔍 YOLO Backbone Output Shape: {x.shape}")  # Debug step
        
        x = x.mean([2, 3])  # Global Average Pooling
        print(f"🔍 Shape After Global Pooling: {x.shape}")  # Debug step

        x = x.view(x.size(0), -1)  # Flatten before classification
        print(f"🔍 Shape Before Classifier: {x.shape}")  # Debug step

        x = self.classifier(x)  # Pass through classifier
        return x

# ✅ Load pretrained YOLO backbone
yolo_base = YOLO("yolo11n.pt")
classifier_model = YOLOClassifier(yolo_base, num_classes=len(class_mapping)).to("cuda")

# ✅ Debug Tensor Flow
for images, labels in train_loader:
    images = images.to("cuda")

    # ✅ Convert labels to numerical indices safely
    labels = [class_mapping.get(label, -1) for label in labels]
    labels = [label for label in labels if label != -1]  # Filter out invalid labels
    labels = torch.tensor(labels, dtype=torch.long).to("cuda")

    optimizer = torch.optim.Adam(classifier_model.parameters(), lr=1e-4)
    criterion = torch.nn.CrossEntropyLoss()

    optimizer.zero_grad()
    outputs = classifier_model(images)  # ✅ Debug model flow
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    print("✅ Successfully processed first batch!")
    break  # Only test one batch for debugging


✅ Verified Class Mapping: {'construction_crane': 0, 'bulldozer': 1, 'forklift': 2, 'excavator': 3, 'cement_mixer': 4, 'dump_truck': 5, 'backhoe': 6, 'loader': 7, 'paver': 8, 'hard_hat': 9, 'safety_vest': 10, 'safety_goggles': 11, 'gloves': 12, 'boots': 13, 'harness': 14, 'respirator_mask': 15, 'scaffolding': 16, 'barricade': 17, 'traffic_cone': 18, 'construction_sign': 19, 'wheelbarrow': 20, 'ladder': 21, 'cables_wiring': 22, 'unstable_structure': 23, 'exposed_wiring': 24, 'falling_debris': 25, 'fire_risk': 26, 'oil_spill': 27}
📊 Found 658 images in train
🔍 Input Image Shape: torch.Size([16, 3, 640, 640])


TypeError: cat() received an invalid combination of arguments - got (Tensor, int), but expected one of:
 * (tuple of Tensors tensors, int dim = 0, *, Tensor out = None)
 * (tuple of Tensors tensors, name dim, *, Tensor out = None)


Testing above and executing below

In [31]:
import torch
import torch.nn as nn
from ultralytics import YOLO

# ✅ Load YOLO11 backbone for classification
class YOLOClassifier(nn.Module):
    def __init__(self, base_model, num_classes):
        super(YOLOClassifier, self).__init__()
        self.base_model = base_model.model.model[:-1]  # Remove detection head
        self.classifier = nn.Linear(1024, num_classes)  # Replace with classification head

    def forward(self, x):
        x = self.base_model(x)  # Extract features
        print(f"🔍 Model Output Shape Before Pooling: {x.shape}")  # Debug step
        
        x = x.mean(dim=[2, 3])  # ✅ Ensure correct tensor reduction
        x = x.view(x.size(0), -1)  # ✅ Flatten before classification
        
        print(f"🔍 Flattened Shape Before Classification: {x.shape}")  # ✅ Debug step

        x = self.classifier(x)
        return x


# ✅ Load pretrained YOLO as feature extractor
yolo_base = YOLO("yolo11n.pt")
classifier_model = YOLOClassifier(yolo_base, num_classes=28).to(device)


In [19]:
import os

# 📂 Paths
base_path = r"E:\yolo_project\mined"

# ✅ Generate class mapping from dataset folders
class_mapping = {category: idx for idx, category in enumerate(os.listdir(base_path))}
print("🔎 Initial Class Mapping:", class_mapping)

# ✅ Scan dataset & verify labels
missing_labels = set()
for split in ["train", "val", "test"]:
    split_path = os.path.join(base_path, split)
    if not os.path.exists(split_path):
        continue

    for category in os.listdir(split_path):
        category_path = os.path.join(split_path, category)
        if not os.path.isdir(category_path):
            continue

        # ✅ Ensure category exists in class_mapping
        if category not in class_mapping:
            missing_labels.add(category)

# 🔄 Auto-Fix Missing Labels
for missing_label in missing_labels:
    class_mapping[missing_label] = len(class_mapping)  # Assign new index

print("✅ Updated Class Mapping:", class_mapping)


🔎 Initial Class Mapping: {'test': 0, 'train': 1, 'val': 2}
✅ Updated Class Mapping: {'test': 0, 'train': 1, 'val': 2, 'construction_crane': 3, 'fire_risk': 4, 'oil_spill': 5, 'gloves': 6, 'safety_goggles': 7, 'forklift': 8, 'ladder': 9, 'respirator_mask': 10, 'construction_sign': 11, 'cement_mixer': 12, 'backhoe': 13, 'paver': 14, 'scaffolding': 15, 'dump_truck': 16, 'wheelbarrow': 17, 'traffic_cone': 18, 'hard_hat': 19, 'barricade': 20, 'loader': 21, 'unstable_structure': 22, 'bulldozer': 23, 'excavator': 24, 'exposed_wiring': 25, 'boots': 26, 'falling_debris': 27, 'cables_wiring': 28, 'harness': 29, 'safety_vest': 30}


In [20]:
# ✅ Generate class mapping ONLY from category folders
class_mapping = {}  # Initialize empty dictionary

for split in ["train", "val", "test"]:
    split_path = os.path.join(base_path, split)
    if not os.path.exists(split_path):
        continue

    for category in os.listdir(split_path):
        category_path = os.path.join(split_path, category)
        if os.path.isdir(category_path):  # ✅ Ensure it's a folder, not a file
            if category not in class_mapping:
                class_mapping[category] = len(class_mapping)  # Assign unique index

print("✅ Final Class Mapping:", class_mapping)


✅ Final Class Mapping: {'construction_crane': 0, 'bulldozer': 1, 'forklift': 2, 'excavator': 3, 'cement_mixer': 4, 'dump_truck': 5, 'backhoe': 6, 'loader': 7, 'paver': 8, 'hard_hat': 9, 'safety_vest': 10, 'safety_goggles': 11, 'gloves': 12, 'boots': 13, 'harness': 14, 'respirator_mask': 15, 'scaffolding': 16, 'barricade': 17, 'traffic_cone': 18, 'construction_sign': 19, 'wheelbarrow': 20, 'ladder': 21, 'cables_wiring': 22, 'unstable_structure': 23, 'exposed_wiring': 24, 'falling_debris': 25, 'fire_risk': 26, 'oil_spill': 27}


In [23]:
# ✅ Build class mapping from all splits
class_mapping = {}

for split in ["train", "val", "test"]:
    split_path = os.path.join(base_path, split)
    if not os.path.exists(split_path):
        continue

    for category in os.listdir(split_path):
        category_path = os.path.join(split_path, category)
        if os.path.isdir(category_path):  # ✅ Ensure it's a folder, not a file
            if category not in class_mapping:
                class_mapping[category] = len(class_mapping)  # Assign unique index

print("✅ Verified Class Mapping:", class_mapping)


✅ Verified Class Mapping: {'construction_crane': 0, 'bulldozer': 1, 'forklift': 2, 'excavator': 3, 'cement_mixer': 4, 'dump_truck': 5, 'backhoe': 6, 'loader': 7, 'paver': 8, 'hard_hat': 9, 'safety_vest': 10, 'safety_goggles': 11, 'gloves': 12, 'boots': 13, 'harness': 14, 'respirator_mask': 15, 'scaffolding': 16, 'barricade': 17, 'traffic_cone': 18, 'construction_sign': 19, 'wheelbarrow': 20, 'ladder': 21, 'cables_wiring': 22, 'unstable_structure': 23, 'exposed_wiring': 24, 'falling_debris': 25, 'fire_risk': 26, 'oil_spill': 27}


In [25]:
# ✅ Verify how many images per category are being detected
category_counts = {}

for split in ["train", "val", "test"]:
    split_path = os.path.join(base_path, split)
    if not os.path.exists(split_path):
        continue

    for category in os.listdir(split_path):
        category_path = os.path.join(split_path, category)
        if os.path.isdir(category_path):  # ✅ Ensure it's a folder
            num_images = len([img for img in os.listdir(category_path) if img.endswith((".jpg", ".png"))])
            category_counts[category] = num_images

print("📊 Dataset Image Count Per Category:", category_counts)


📊 Dataset Image Count Per Category: {'construction_crane': 10, 'bulldozer': 6, 'forklift': 7, 'excavator': 6, 'cement_mixer': 6, 'dump_truck': 7, 'backhoe': 7, 'loader': 6, 'paver': 8, 'hard_hat': 7, 'safety_vest': 6, 'safety_goggles': 6, 'gloves': 8, 'boots': 6, 'harness': 7, 'respirator_mask': 7, 'scaffolding': 6, 'barricade': 8, 'traffic_cone': 7, 'construction_sign': 6, 'wheelbarrow': 7, 'ladder': 7, 'cables_wiring': 6, 'unstable_structure': 7, 'exposed_wiring': 7, 'falling_debris': 6, 'fire_risk': 7, 'oil_spill': 7}


In [32]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier_model.parameters(), lr=1e-4)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)

for epoch in range(50):
    total_loss = 0

    # Map class names to numerical indices
    class_mapping = {category: idx for idx, category in enumerate(os.listdir(base_path))}

    for images, labels in train_loader:
        print(f"📊 Raw Labels from DataLoader: {labels}")  # ✅ Debug step
        break  # Only print one batch for readability
    
    for images, labels in train_loader:
        images = images.to(device)

        # ✅ Convert labels to numerical indices safely
        labels = [class_mapping.get(label, -1) for label in labels]  # Get index or -1 if missing

        # ✅ Filter out unmapped labels to avoid KeyErrors
        labels = [label for label in labels if label != -1]
        
        labels = torch.tensor(labels, dtype=torch.long).to(device)

        optimizer.zero_grad()
        outputs = classifier_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()



    print(f"Epoch {epoch+1}: Avg Loss = {total_loss / len(train_loader):.4f}")


📊 Raw Labels from DataLoader: ('construction_crane', 'construction_crane', 'construction_crane', 'construction_crane', 'construction_crane', 'construction_crane', 'construction_crane', 'construction_crane', 'construction_crane', 'construction_crane', 'construction_crane', 'construction_crane', 'construction_crane', 'construction_crane', 'construction_crane', 'construction_crane')


TypeError: cat() received an invalid combination of arguments - got (Tensor, int), but expected one of:
 * (tuple of Tensors tensors, int dim = 0, *, Tensor out = None)
 * (tuple of Tensors tensors, name dim, *, Tensor out = None)
